# Start-to-Finish Cartesian Wave Project

This notebook runs the packaged Cartesian wave-equation generator, inspects the generated project, builds when local tools are available, and reads convergence diagnostics.

## Table of Contents

1. [Workflow](#Workflow)
2. [Generator discovery](#Generator-discovery)
3. [Temporary project workspace](#Temporary-project-workspace)
4. [Generate and inspect](#Generate-and-inspect)
5. [Build and run](#Build-and-run)
6. [Convergence diagnostics](#Convergence-diagnostics)
7. [Related notebooks](#Related-notebooks)

## Workflow

The generator command is:

```bash
python -m nrpy.examples.wave_equation_cartesian
```

The command creates a generated C project. This notebook runs it in a temporary workspace so existing learner projects are not overwritten.

## Generator Discovery

Discovery uses `importlib.util.find_spec` so the generator is not imported at notebook startup.

In [ ]:
import importlib.util
import subprocess
import sys
import tempfile
from pathlib import Path

import nrpy


print("Imported nrpy from:", nrpy.__file__)

generator_name = "nrpy.examples.wave_equation_cartesian"
generator_spec = importlib.util.find_spec(generator_name)
print("Generator discovered:", generator_spec is not None)

## Temporary Project Workspace

The temporary workspace object is kept alive across generation, build, run, and diagnostic-inspection cells.

In [ ]:
workspace_manager = tempfile.TemporaryDirectory(prefix="nrpy_wave_cartesian_")
workspace_path = Path(workspace_manager.name)
project_path = workspace_path / "project" / "wave_equation_cartesian"

print("Temporary workspace created.")
print("Project directory exists before generation:", project_path.exists())

## Generate and Inspect

The generator produces project files, build rules, generated C functions, and runtime parameter data. The inspection below prints a compact inventory.

In [ ]:
generation_completed = False
if generator_spec is None:
    raise RuntimeError(f"Required generator not found: {generator_name}")
else:
    generate = subprocess.run(
        [sys.executable, "-m", generator_name],
        cwd=workspace_path,
        text=True,
        capture_output=True,
        timeout=180,
    )
    print("Generator return code:", generate.returncode)
    if generate.stdout.strip():
        print("\n".join(generate.stdout.splitlines()[:16]))
    if generate.returncode != 0:
        if generate.stderr.strip():
            print("Generator stderr excerpt:")
            print("\n".join(generate.stderr.splitlines()[:12]))
        raise RuntimeError("Cartesian wave generator failed.")
    generation_completed = generate.returncode == 0

if project_path.exists():
    project_files = sorted(
        path.relative_to(project_path)
        for path in project_path.rglob("*")
        if path.is_file()
    )
    print("Generated file count:", len(project_files))
    print("First generated files:")
    for relpath in project_files[:18]:
        print(" ", relpath)

    for relpath in project_files:
        if relpath.name in {"Makefile", "main.c", "wave_equation_cartesian.c"}:
            excerpt = (project_path / relpath).read_text(encoding="utf-8", errors="replace").splitlines()[:10]
            print(f"\nExcerpt from {relpath}:")
            print("\n".join(excerpt))
            break
else:
    print("Generated project directory is not present.")

## Build and Run

The project build requires `make` and a C compiler. If they are unavailable, the notebook prints the commands a learner can run later.

In [ ]:
import shutil


make_tool = shutil.which("make")
compiler_tool = shutil.which("cc") or shutil.which("gcc") or shutil.which("clang")
can_build = project_path.exists() and make_tool is not None and compiler_tool is not None

print("make available:", make_tool is not None)
print("C compiler available:", compiler_tool is not None)

build_completed = False
build_skipped_for_tools = False
run_completed = False
if not project_path.exists():
    raise RuntimeError("Build cannot proceed because no generated project is available.")
elif not can_build:
    build_skipped_for_tools = True
    print("Build skipped. From the generated project directory, run: make -j2")
    print("Then run: ./wave_equation_cartesian")
    print("For a convergence rerun, run: ./wave_equation_cartesian 2.0")
else:
    build = subprocess.run(
        ["make", "-j2"],
        cwd=project_path,
        text=True,
        capture_output=True,
        timeout=180,
    )
    print("Build return code:", build.returncode)
    if build.stdout.strip():
        print("\n".join(build.stdout.splitlines()[:16]))
    if build.returncode != 0:
        if build.stderr.strip():
            print("Build stderr excerpt:")
            print("\n".join(build.stderr.splitlines()[:16]))
        raise RuntimeError("Generated Cartesian wave project build failed.")
    build_completed = build.returncode == 0

    executable = project_path / "wave_equation_cartesian"
    if build_completed and not executable.exists():
        raise RuntimeError("Generated Cartesian wave executable was not created.")
    for args in ([], ["2.0"]):
        label = "default" if not args else "convergence factor 2.0"
        run = subprocess.run(
            [f"./{executable.name}", *args],
            cwd=project_path,
            text=True,
            capture_output=True,
            timeout=180,
        )
        print(f"Run ({label}) return code:", run.returncode)
        if run.stdout.strip():
            print("\n".join(run.stdout.splitlines()[:12]))
        if run.returncode != 0 and run.stderr.strip():
            print("Run stderr excerpt:")
            print("\n".join(run.stderr.splitlines()[:12]))
        if run.returncode != 0:
            raise RuntimeError(f"Generated Cartesian wave run failed: {label}")
        run_completed = True

## Convergence Diagnostics

Successful runs write diagnostic text files. The second run, with convergence factor `2.0`, should produce a file whose name records that factor.

In [ ]:
if project_path.exists():
    diagnostics = sorted(project_path.glob("out0d-conv_factor*.txt"))
    if diagnostics:
        for diagnostic in diagnostics:
            print("Diagnostic file:", diagnostic.name)
            print("\n".join(diagnostic.read_text(encoding="utf-8", errors="replace").splitlines()[:10]))
    elif run_completed:
        raise RuntimeError("No convergence diagnostic files were found after successful runs.")
    elif build_skipped_for_tools:
        print("No convergence diagnostics are available because build/run work was skipped.")
    else:
        print("No convergence diagnostic files were found.")
else:
    print("No generated project is available for diagnostic inspection.")

## Related Notebooks

[Wave equation and C code generation](wave_equation_and_c_codegen.ipynb) explains the symbolic expressions used in the project. [Boundary conditions and convergence](../2-numerical_methods/boundary_conditions_and_convergence.ipynb) discusses ghost zones, boundary fills, and convergence reruns at the workflow level.